# 01 - arXiv scraper
Goal: collect 500+ paper metadata records (cs.LG + cs.AI, 2022–2026)

Output: data/raw/raw_papers.jsonl + data/raw/pdfs/

In [2]:
# Run this as the first cell in every Colab notebook
!pip install -q transformers peft trl faiss-cpu arxiv \
    PyMuPDF gradio sentence-transformers evaluate \
    datasets bitsandbytes accelerate jsonlines

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/arxiv-llm-project'
DATA_RAW   = f'{DRIVE_ROOT}/data/raw'
DATA_PROC  = f'{DRIVE_ROOT}/data/processed'

import os
os.makedirs(DATA_RAW + '/pdfs', exist_ok=True)
os.makedirs(DATA_PROC, exist_ok=True)
print("Environment ready ✓")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.8 MB/s eta 0:00:00
Mounted at /content/drive
Environment ready ✓


In [3]:
import arxiv
import collections
import time
from tqdm import tqdm

# Combined categories to fetch everything in one pass per year
# This is much more API-friendly
YEARS = [2022, 2023, 2024, 2025, 2026]
LIMIT_PER_YEAR = 5000

def fetch_safe():
    # We increase delay_seconds to 5 to be very respectful of the API
    client = arxiv.Client(
        page_size=100,
        delay_seconds=5,
        num_retries=10
    )

    all_results = []

    for year in YEARS:
        print(f"\n--- Fetching Year {year} ---")

        # We query BOTH categories at once using OR
        query = f"(cat:cs.LG OR cat:cs.AI) AND submittedDate:[{year}01010000 TO {year}12312359]"

        search = arxiv.Search(
            query=query,
            max_results=LIMIT_PER_YEAR,
            sort_by=arxiv.SortCriterion.SubmittedDate,
            sort_order=arxiv.SortOrder.Ascending # Changed to Ascending
        )

        try:
            results_gen = client.results(search)
            for r in tqdm(results_gen, total=LIMIT_PER_YEAR):
                all_results.append({
                    "paper_id": r.get_short_id(),
                    "title": r.title.strip(),
                    "abstract": r.summary.strip().replace("\n"," "),
                    "year": r.published.year,
                    "category": r.primary_category, # Gets the real primary category
                    "url": r.entry_id
                })
        except Exception as e:
            print(f"Error in {year}: {e}")
            print("Taking a 60-second break...")
            time.sleep(60) # Longer break if we hit a wall
            continue

    return all_results

raw_data = fetch_safe()

# Deduplicate
seen, deduped = set(), []
for p in raw_data:
    if p["paper_id"] not in seen:
        seen.add(p["paper_id"])
        deduped.append(p)

# Print Summary
years_found = collections.Counter(p["year"] for p in deduped)
print("\nFinal Results:", dict(sorted(years_found.items())))


--- Fetching Year 2022 ---


100%|██████████| 5000/5000 [10:02<00:00,  8.30it/s]



--- Fetching Year 2023 ---


100%|██████████| 5000/5000 [04:40<00:00, 17.80it/s]



--- Fetching Year 2024 ---


...


--- Fetching Year 2025 ---


100%|██████████| 5000/5000 [04:48<00:00, 17.31it/s]



--- Fetching Year 2026 ---


100%|██████████| 5000/5000 [04:43<00:00, 17.65it/s]


Final Results: {2022: 5000, 2023: 5000, 2024: 5000, 2025: 5000, 2026: 5000}


In [4]:
import collections
import json

JSONL = "papers_data.jsonl"

with open(JSONL, "w") as f:
    for p in deduped:
        f.write(json.dumps(p) + "\n")

# Use .get() or check keys to avoid KeyErrors if some records are missing fields
counts = collections.Counter(p.get("category", "unknown") for p in deduped)
years  = collections.Counter(p.get("year", "unknown") for p in deduped)

print(f"Written {len(deduped)} records to {JSONL}")
print("By category:", dict(counts))
print("By year:",     dict(years))

Written 25000 records to papers_data.jsonl
By category: {'eess.IV': 463, 'math.AT': 3, 'cs.NE': 218, 'cs.CV': 2691, 'cs.LG': 9518, 'cs.SE': 397, 'cs.CL': 2404, 'math.NA': 92, 'cs.AI': 2189, 'cs.RO': 538, 'eess.SY': 141, 'stat.ML': 952, 'cs.DS': 56, 'cond-mat.mtrl-sci': 57, 'cs.GT': 118, 'math.OC': 234, 'q-fin.ST': 44, 'eess.SP': 272, 'physics.comp-ph': 26, 'physics.flu-dyn': 48, 'cs.DB': 63, 'econ.GN': 15, 'cs.CR': 663, 'cs.SI': 132, 'math.AP': 9, 'cs.IR': 385, 'cond-mat.str-el': 8, 'cs.NI': 167, 'cs.SD': 334, 'q-bio.GN': 37, 'q-bio.QM': 77, 'cs.AR': 84, 'cs.MA': 128, 'astro-ph.CO': 10, 'quant-ph': 157, 'eess.AS': 190, 'stat.ME': 76, 'astro-ph.GA': 10, 'gr-qc': 7, 'cs.DC': 146, 'cs.HC': 354, 'cs.GR': 32, 'stat.AP': 30, 'math.ST': 50, 'cs.CY': 370, 'q-fin.CP': 15, 'cs.CE': 45, 'physics.ao-ph': 33, 'astro-ph.EP': 26, 'math.DS': 13, 'physics.chem-ph': 38, 'cond-mat.stat-mech': 9, 'cond-mat.soft': 8, 'math.PR': 15, 'cs.ET': 31, 'cs.IT': 114, 'cs.MS': 6, 'q-bio.NC': 65, 'physics.med-ph': 17

In [ ]:
import requests, time, os
from pathlib import Path
from tqdm.auto import tqdm

# --- CONFIGURATION ---
PDF_DIR = "drive/MyDrive/research_papers"
PDF_LIMIT = 2000
DELAY = 3  # ArXiv is strict; keep this at 3+ seconds to avoid IP bans
os.makedirs(PDF_DIR, exist_ok=True)

ok = skipped = failed = 0

for paper in tqdm(deduped[:PDF_LIMIT], desc="Downloading PDFs"):
    paper_id = paper['paper_id']
    dest = Path(PDF_DIR) / f"{paper_id}.pdf"

    if dest.exists():
        skipped += 1
        continue

    # FIX: Construct URL if 'pdf_url' is missing
    # ArXiv PDF pattern: https://arxiv.org/pdf/{id}.pdf
    url = paper.get("pdf_url") or f"https://arxiv.org/pdf/{paper_id}.pdf"

    try:
        # Use headers to look like a real browser
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        r = requests.get(url, timeout=30, headers=headers)
        r.raise_for_status()

        dest.write_bytes(r.content)
        ok += 1
    except Exception as e:
        # print(f"  ✗ {paper_id}: {e}") # Debug line
        failed += 1

    time.sleep(DELAY)

print(f"\nDone — new: {ok}, skipped: {skipped}, failed: {failed}")